# 03 CPU・GPU・TPU（JAX を中心に）

## 1. 研究目的（RQ3・Advanced A）

Google Colab の CPU・GPU・TPU ランタイムで、**要素ごとの演算・ベクトル総和・行列積**の性能を比べます。

- **同じ JAX 関数**を CPU と、ランタイムのアクセラレータ（GPU または TPU）で実行し、「**デバイスの差**」を見る
- NumPy（CPU）・PyTorch（CPU / GPU）とも比べ、「**ライブラリ（フレームワーク）の差**」を見る
- 時間を **データ生成・データ転送・初回（コンパイル込み）・コンパイル・ウォームアップ後の計算・結果の取り出し** に分けて測る

> ### ⚠️ 最初に読む
> - TPU の実験は **Advanced Research** です。GPU・TPU が使えなくても、CPU で Notebook を最後まで実行できます。
> - 表やグラフに出るのは、**このランタイムで実際に測定できたデバイスだけ**です。使えなかったデバイスの値は作りません。
> - 表示される数値は、Colab の共有マシンでの 1 回の測定です。割り当てられた機種・混雑状況で変わります。

## 2. 理論的背景

### TPU と JAX

- **TPU**（Tensor Processing Unit）は Google が設計した、**行列積専用の回路**（systolic array）を持つ装置です。
- **JAX** は NumPy とほぼ同じ書き方（`jax.numpy`）で、CPU・GPU・TPU のどれでも同じ関数を実行できるライブラリです。
- `jax.jit` は関数を **XLA コンパイラ**でコンパイルします。最初の呼び出しで**コンパイル**（時間がかかる）し、2 回目以降は同じ入力の形（**shape**・dtype）ならコンパイル済みのものを使います。shape が変わると再コンパイルします。
- JAX も GPU・TPU では**非同期**に実行されるので、`result.block_until_ready()` で終わるのを待ってから時計を止めます。

### 測る時間の区別

| 記録名 | 意味 |
| --- | --- |
| `data_gen_s` | NumPy で入力データを作る時間（CPU） |
| `transfer_s` | `jax.device_put`（PyTorch は `.to(device)`）でデバイスへ送る時間。PyTorch の CPU では実際にはコピーしないことがあるので、ほぼ 0 になる |
| `first_call_s` | `jit` してすぐ呼んだ 1 回目（トレース＋**コンパイル**＋実行） |
| `compile_s` | コンパイルだけの時間（`jax.jit(f).lower(...).compile()`） |
| `compute_s` | コンパイル済みの関数の、ウォームアップ後の実行時間（中央値）＝**計算本体** |
| `fetch_s` | 結果を NumPy 配列として CPU に取り出す時間 |

### 3 つの計算の性格

| 計算 | 計算量 | データ量 | 1 データあたりの計算 | 予想 |
| --- | --- | --- | --- | --- |
| 要素ごとの演算 $a b + c$ | $O(n)$ | $4n$ | 少ない | メモリの読み書きの速さで決まりやすい |
| 総和 $\sum a_i$ | $O(n)$ | $n$ | 少ない（足し合わせの順序に依存関係） | 並列化には「木構造の足し算（reduction）」が必要 |
| 行列積 $AB$ | $O(n^3)$ | $3n^2$ | 多い | GPU・TPU が最も得意 |

## 3. 実験環境（最初に必ず実行）

このセルは、E2 リポジトリの `.py`（`cpu_basics.py`、`numpy_benchmark.py` など）を import できるように、
**作業ディレクトリ（E2 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E2 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E2 フォルダを ZIP にして `/content` にアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/`（教科書のコード）は**公開リポジトリには含めない**約束です。
> `textbook/` がないと `algorithm_benchmark.py` の import が失敗します（Notebook 04・05 の一部）。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e2-computing-performance"   # リポジトリの中の E2 フォルダ（ルートが E2 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e2_root(p):
    """cpu_basics.py と benchmark_utils.py があるフォルダを E2 のルートとみなす"""
    return (Path(p) / "cpu_basics.py").exists() and (Path(p) / "benchmark_utils.py").exists()


def find_e2_root():
    # 1) ローカル：notebooks/ の中、または E2 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e2_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e2_upload")
    for hit in glob.glob("/content/e2_upload/**/cpu_basics.py", recursive=True):
        if is_e2_root(Path(hit).parent):
            return Path(hit).parent
    return None


E2_ROOT = find_e2_root()
if E2_ROOT is None:
    raise RuntimeError("E2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E2_ROOT)
if str(E2_ROOT) not in sys.path:
    sys.path.insert(0, str(E2_ROOT))
print("E2_ROOT:", E2_ROOT)
if not (E2_ROOT / "textbook" / "bsearch.py").exists():
    print("⚠️ textbook/ がありません。algorithm_benchmark.py の import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.6), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は「実装（ライブラリ）」ごとに固定する。デバイス（CPU / GPU / TPU）は線の種類と記号で区別する
COLORS = {"Python": "#eb6834", "NumPy": "#2a78d6", "PyTorch": "#1baf7a", "JAX": "#6250d6",
          "TensorFlow": "#eda100", "neutral": "#8a8985"}
DEVICE_STYLE = {"CPU": ("-", "o"), "GPU": ("--", "s"), "TPU": ("-.", "^")}

### 3-1. 実行環境の検出

Python・NumPy・PyTorch・JAX のバージョン、CPU、GPU・TPU の有無を表示します（ここでは測定しません）。
Colab には NumPy・PyTorch・JAX が標準で入っています。**GPU・TPU 用のパッケージをこの Notebook から自動でインストールすることはしません**（ランタイムに合わないものを入れると動かなくなるため）。

In [ ]:
import platform

def show_environment():
    """Python・ライブラリのバージョン、CPU、GPU・TPU の有無を表示する（測定はしない）"""
    info = {"Python": platform.python_version(), "Platform": platform.platform()}
    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as f:
            cpu = next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        pass
    info["CPU"] = f"{cpu} ({os.cpu_count()} logical cores)"
    try:
        import numpy
        info["NumPy"] = numpy.__version__
    except ImportError:
        info["NumPy"] = "not installed"
    try:
        import torch
        info["PyTorch"] = torch.__version__
        info["CUDA GPU"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available"
    except ImportError:
        info["PyTorch"] = "not installed"
        info["CUDA GPU"] = "unknown (PyTorch not installed)"
    try:
        import jax
        info["JAX"] = jax.__version__
        info["JAX devices"] = ", ".join(f"{d.platform}:{d.device_kind}" for d in jax.devices())
        info["TPU"] = "available" if any(d.platform == "tpu" for d in jax.devices()) else "not available"
    except ImportError:
        info["JAX"] = "not installed"
        info["TPU"] = "unknown (JAX not installed)"
    except Exception as e:   # TPU の初期化に失敗した場合など
        info["JAX"] = info.get("JAX", "installed") + f" (device error: {e})"
        info["TPU"] = "not available"
    for k, v in info.items():
        print(f"{k:12}: {v}")
    return info

ENV = show_environment()

In [ ]:
import time
import csv
import numpy as np
from benchmark_utils import measure, format_table

try:
    import jax
    import jax.numpy as jnp
    HAS_JAX = True
except ImportError:
    HAS_JAX = False
    print("JAX がインストールされていないので、JAX の部分はスキップします（ローカルの CPU なら pip install jax）")

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

# 実際に使う計算デバイスの一覧（同じ JAX 関数を、CPU と アクセラレータで実行する）
JAX_TARGETS = []
if HAS_JAX:
    jax.config.update("jax_default_matmul_precision", "highest")   # TPU でも float32 相当の精度で比べる
    JAX_TARGETS.append(("CPU", jax.devices("cpu")[0]))
    default = jax.devices()[0]
    if default.platform != "cpu":
        JAX_TARGETS.append(({"gpu": "GPU", "cuda": "GPU", "tpu": "TPU"}.get(default.platform, default.platform.upper()), default))
TORCH_TARGETS = (["cpu"] + (["cuda"] if torch.cuda.is_available() else [])) if HAS_TORCH else []

print("JAX    :", [(name, str(d)) for name, d in JAX_TARGETS] or "なし")
print("PyTorch:", TORCH_TARGETS or "なし")

### 3-2. 実験条件と安全上限

無料の Colab でも数分で終わる小さめの条件です。上限を超える条件は**実行しません**。

In [ ]:
SEED = 0
DTYPE = np.float32
WARMUP, REPEATS = 3, 10

VECTOR_SIZES = [10_000, 100_000, 1_000_000]   # 要素ごとの演算・総和の n
MATRIX_SIZES = [128, 512, 1024]               # 行列積の n（n×n）
MAX_VECTOR_N = 20_000_000                      # 約 80 MB × 4 配列（float32）
MAX_MATRIX_N = 2048                            # 2048×2048 float32 = 16 MB × 3 行列
TIME_BUDGET_S = 300                            # 全測定の時間の目安（秒）

## 4. 使用するコード

同じ計算を NumPy・JAX・PyTorch で書きます。**JAX の関数は 1 つだけ**で、それを CPU とアクセラレータの両方で実行します。

In [ ]:
# 計算の定義（どれも同じ数学的計算）
NP_OPS = {
    "elementwise": lambda a, b, c: a * b + c,
    "sum": lambda a: np.sum(a),
    "matmul": lambda A, B: A @ B,
}
if HAS_JAX:
    JAX_OPS = {
        "elementwise": lambda a, b, c: a * b + c,
        "sum": lambda a: jnp.sum(a),
        "matmul": lambda A, B: A @ B,
    }
if HAS_TORCH:
    TORCH_OPS = {
        "elementwise": lambda a, b, c: a * b + c,
        "sum": lambda a: torch.sum(a),
        "matmul": lambda A, B: A @ B,
    }


def make_data(op, n, seed=SEED):
    """op 用の入力（NumPy 配列）を作る。全フレームワークで同じデータを使う"""
    rng = np.random.default_rng(seed)
    if op == "matmul":
        if n > MAX_MATRIX_N:
            raise ValueError("行列サイズが安全上限を超えています")
        return [rng.standard_normal((n, n)).astype(DTYPE) for _ in range(2)]
    if n > MAX_VECTOR_N:
        raise ValueError("ベクトルの長さが安全上限を超えています")
    k = 3 if op == "elementwise" else 1
    return [rng.standard_normal(n).astype(DTYPE) for _ in range(k)]


def check(result, expected, op, n):
    """float32 の丸め誤差を考慮した一致の確認（総和・行列積は足し合わせる数に応じて許容誤差を広げる）"""
    scale = {"elementwise": 1, "sum": np.sqrt(n), "matmul": np.sqrt(n)}[op]
    return bool(np.allclose(np.asarray(result, dtype=np.float64), np.asarray(expected, dtype=np.float64),
                            rtol=1e-4, atol=1e-4 * scale))


print("定義した計算:", list(NP_OPS))

## 5. 実行例

### 5-1. 計測関数

`bench_jax` は 1 つの計算・サイズ・デバイスについて、上の表の 6 つの時間を測ります。
JAX は同じ関数のコンパイル結果を覚えている（キャッシュする）ので、コンパイルを含む時間を測るときは、毎回**新しい関数オブジェクト**（`lambda *x: f(*x)`）で包みます。

In [ ]:
results = []
T_START = time.perf_counter()


def bench_numpy(op, n):
    t0 = time.perf_counter(); args = make_data(op, n); t_gen = time.perf_counter() - t0
    r = measure(lambda: NP_OPS[op](*args), repeats=REPEATS, warmup=WARMUP)
    return {"framework": "NumPy", "device": "CPU", "op": op, "n": n, "dtype": np.dtype(DTYPE).name,
            "data_gen_s": t_gen, "compute_s": r["median"], "compute_stdev_s": r["stdev"]}, args


def bench_jax(op, n, label, device, expected):
    f = JAX_OPS[op]
    t0 = time.perf_counter(); host = make_data(op, n); t_gen = time.perf_counter() - t0

    t0 = time.perf_counter()
    args = [jax.device_put(x, device) for x in host]
    for x in args:
        x.block_until_ready()
    t_transfer = time.perf_counter() - t0

    t0 = time.perf_counter()
    jax.jit(lambda *x: f(*x))(*args).block_until_ready()          # トレース＋コンパイル＋実行
    t_first = time.perf_counter() - t0

    t0 = time.perf_counter()
    compiled = jax.jit(lambda *x: f(*x)).lower(*args).compile()     # コンパイルだけ
    t_compile = time.perf_counter() - t0

    r = measure(lambda: compiled(*args), repeats=REPEATS, warmup=WARMUP, sync=lambda y: y.block_until_ready())

    out = compiled(*args).block_until_ready()
    t0 = time.perf_counter()
    out_host = np.asarray(out)                                      # CPU に取り出す
    t_fetch = time.perf_counter() - t0

    return {"framework": "JAX", "device": label, "op": op, "n": n, "dtype": np.dtype(DTYPE).name,
            "data_gen_s": t_gen, "transfer_s": t_transfer, "first_call_s": t_first, "compile_s": t_compile,
            "compute_s": r["median"], "compute_stdev_s": r["stdev"], "fetch_s": t_fetch,
            "match_numpy": check(out_host, expected, op, n)}


def bench_torch(op, n, dev, host, expected):
    sync = (lambda _r=None: torch.cuda.synchronize()) if dev == "cuda" else None
    t0 = time.perf_counter()
    args = [torch.as_tensor(x).to(dev) for x in host]
    if sync: sync()
    t_transfer = time.perf_counter() - t0
    f = TORCH_OPS[op]
    r = measure(lambda: f(*args), repeats=REPEATS, warmup=WARMUP, sync=sync)
    out = f(*args)
    if sync: sync()
    t0 = time.perf_counter(); out_host = out.cpu().numpy(); t_fetch = time.perf_counter() - t0
    return {"framework": "PyTorch", "device": "GPU" if dev == "cuda" else "CPU", "op": op, "n": n,
            "dtype": np.dtype(DTYPE).name, "transfer_s": t_transfer, "compute_s": r["median"],
            "compute_stdev_s": r["stdev"], "fetch_s": t_fetch, "match_numpy": check(out_host, expected, op, n)}


def run(op, n):
    if time.perf_counter() - T_START > TIME_BUDGET_S:
        print(f"skip (time budget): {op} n={n}")
        return
    row, host = bench_numpy(op, n)
    results.append(row)
    expected = NP_OPS[op](*host)
    for label, dev in JAX_TARGETS:
        results.append(bench_jax(op, n, label, dev, expected))
    for dev in TORCH_TARGETS:
        results.append(bench_torch(op, n, dev, host, expected))

### 5-2. JIT の効果（小さな例）

`jit` しないと、JAX は演算を 1 つずつ実行します（`a * b` → `+ c`）。`jit` すると XLA が複数の演算を**1 つにまとめて（融合して）**実行できます。

In [ ]:
if HAS_JAX:
    a, b, c = [jax.device_put(x, JAX_TARGETS[0][1]) for x in make_data("elementwise", 1_000_000)]
    g = lambda a, b, c: jnp.sin(a) * b + jnp.cos(c) * a    # いくつかの演算を組み合わせた計算
    g_jit = jax.jit(g)
    g_jit(a, b, c).block_until_ready()                     # コンパイルを済ませておく
    sync = lambda y: y.block_until_ready()
    t_eager = measure(lambda: g(a, b, c), repeats=REPEATS, warmup=WARMUP, sync=sync)["median"]
    t_jit = measure(lambda: g_jit(a, b, c), repeats=REPEATS, warmup=WARMUP, sync=sync)["median"]
    assert np.allclose(np.asarray(g(a, b, c)), np.asarray(g_jit(a, b, c)), rtol=1e-5, atol=1e-6)
    print(f"JAX CPU, n = 1,000,000: without jit {t_eager * 1e3:.2f} ms, with jit {t_jit * 1e3:.2f} ms")

### 5-3. ベンチマークの実行

In [ ]:
for n in VECTOR_SIZES:
    run("elementwise", n)
    run("sum", n)
for n in MATRIX_SIZES:
    run("matmul", n)

print(f"{len(results)} rows, {time.perf_counter() - T_START:.1f} s")
assert all(r.get("match_numpy", True) for r in results), "NumPy と結果が一致しない行があります"
print(format_table(results, [("framework", "framework"), ("device", "device"), ("op", "op"), ("n", "n"),
                             ("transfer_s", "transfer"), ("first_call_s", "first call"),
                             ("compile_s", "compile"), ("compute_s", "compute"), ("fetch_s", "fetch"),
                             ("match_numpy", "match")]))

### 5-4. 結果の保存

ランタイム（CPU・GPU・TPU）ごとに別のファイル名で保存します。Colab ではランタイムを切り替えると**ファイルが消える**ので、ダウンロードするか Google Drive にコピーしてください。

In [ ]:
from benchmark_utils import save_csv
tag = "tpu" if any(l == "TPU" for l, _ in JAX_TARGETS) else ("gpu" if "cuda" in TORCH_TARGETS or any(l == "GPU" for l, _ in JAX_TARGETS) else "cpu")
os.makedirs("notebooks/results", exist_ok=True)
out_path = save_csv(results, f"notebooks/results/03_cpu_gpu_tpu_{tag}.csv")
print("saved:", out_path)
# Colab でダウンロードする場合：
# from google.colab import files; files.download(out_path)

## 6. 結果の可視化

線の色が**ライブラリ**、線の種類と記号が**デバイス**です（実線・丸＝CPU、破線・四角＝GPU、一点鎖線・三角＝TPU）。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, op in zip(axes, ["elementwise", "sum", "matmul"]):
    for fw, dev in sorted({(r["framework"], r["device"]) for r in results if r["op"] == op}):
        pts = sorted((r["n"], r["compute_s"]) for r in results
                     if r["op"] == op and r["framework"] == fw and r["device"] == dev)
        xs, ys = zip(*pts)
        ls, mk = DEVICE_STYLE.get(dev, ("-", "o"))
        ax.loglog(xs, np.array(ys) * 1e3, ls, marker=mk, ms=6, color=COLORS[fw], label=f"{fw} / {dev}")
    ax.set_title(f"{op}: compute time"); ax.set_xlabel("n (matrix: n x n)" if op == "matmul" else "n")
axes[0].set_ylabel("compute time [ms]"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 6-2. JAX の時間の内訳（各計算の最大サイズ）：初回（コンパイル込み）・コンパイル・計算本体・転送・取り出し
jax_rows = [r for r in results if r["framework"] == "JAX"]
if jax_rows:
    sel = []
    for op in ["elementwise", "sum", "matmul"]:
        rs = [r for r in jax_rows if r["op"] == op]
        if rs:
            nmax = max(r["n"] for r in rs)
            sel += [r for r in rs if r["n"] == nmax]
    keys = [("transfer_s", "transfer", "#8a8985"), ("first_call_s", "first call (incl. compile)", "#eb6834"),
            ("compile_s", "compile only", "#eda100"), ("compute_s", "compute (warm median)", COLORS["JAX"]),
            ("fetch_s", "fetch to host", "#2a78d6")]
    width = 0.16
    plt.figure(figsize=(12, 3.8))
    for k, (key, name, color) in enumerate(keys):
        plt.bar(np.arange(len(sel)) + (k - 2) * width, [r[key] * 1e3 for r in sel], width, color=color, label=name)
    plt.xticks(range(len(sel)), [f"{r['op']}\n{r['device']} n={r['n']:,}" for r in sel], fontsize=8)
    plt.yscale("log"); plt.ylabel("time [ms] (log)"); plt.title("JAX: where does the time go?")
    plt.legend(fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.25))
    plt.tight_layout(); plt.show()
else:
    print("JAX の結果がないので、このグラフは表示しません。")

### 6-3. 複数のランタイムの結果をまとめる（任意）

CPU・GPU・TPU の各ランタイムで保存した CSV を `notebooks/results/` に置いてから実行します。**存在するファイルだけ**を読みます。

In [ ]:
merged = []
for p in sorted(glob.glob("notebooks/results/03_cpu_gpu_tpu_*.csv")):
    with open(p) as f:
        merged += [dict(r, runtime=os.path.basename(p)) for r in csv.DictReader(f)]
print("files:", sorted({r["runtime"] for r in merged}))
# TODO：merged から、同じ JAX 関数の compute_s を CPU・GPU・TPU で比べるグラフを作る

### 参考：TensorFlow による TPU の行列積（参考 Notebook 由来・任意・未検証）

参考 Notebook（`A_cpu_gpu_tpu_benchmark/starter.ipynb`）は、TPU を **TensorFlow の `TPUStrategy`** で使っていました。
この教材では JAX を中心にしますが、比較のために同じ方法のセルを残しています（既定では実行しません）。

- 実行するには `RUN_TENSORFLOW_TPU = True` にします。TensorFlow が入っていない、または TPU に接続できない場合はスキップします。
- Colab の TPU ランタイムの仕組みは変わることがあり、この方法が現在のランタイムで動くかは**検証していません**。
- 参考 Notebook からの変更点：コンパイル・初回を除くため**ウォームアップ後に複数回**測り、結果の取り出し（`.numpy()`）と分けています。

In [ ]:
RUN_TENSORFLOW_TPU = False   # True にすると試す（Advanced、未検証）

if RUN_TENSORFLOW_TPU:
    try:
        import tensorflow as tf
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        strategy = tf.distribute.TPUStrategy(resolver)
        SIZE = 1024
        with strategy.scope():
            tx = tf.random.uniform((SIZE, SIZE), dtype=tf.float32)
            ty = tf.random.uniform((SIZE, SIZE), dtype=tf.float32)
        tf_matmul = tf.function(lambda: tf.matmul(tx, ty))
        tf_matmul().numpy()                         # 初回（コンパイル）
        r = measure(lambda: tf_matmul(), repeats=REPEATS, warmup=WARMUP, sync=lambda y: y.numpy())
        print(f"TensorFlow TPU matmul {SIZE}x{SIZE}: median {r['median'] * 1e3:.3f} ms（取り出しを含む）")
    except Exception as e:
        print("TensorFlow TPU の実験はスキップしました:", e)
else:
    print("TensorFlow TPU のセルは実行していません（RUN_TENSORFLOW_TPU = False）")

## 7. 結果の解釈（読み方のヒント）

- **同じ JAX 関数で CPU と アクセラレータを比べる**と、ライブラリの違いを除いた「デバイスの差」に近いものが見られます。
- **NumPy と JAX（どちらも CPU）**の差は、デバイスではなく**ライブラリ・コンパイルの差**です。
- **初回とコンパイル**：JAX の初回はコンパイルを含むので、計算本体の何十〜何千倍もかかることがあります。同じ shape で何度も実行する場合にだけ、コンパイルの時間は「元が取れ」ます。
- **要素ごとの演算・総和**は計算が少なくデータが多いので、メモリの速さと転送が効きます。**行列積**は計算が多く、GPU・TPU の効果が出やすい。
- **転送と取り出し**：計算本体が速くても、データを送る・取り出す時間を含めると、CPU の方が速いことがあります。

> - アクセラレータを使っても、アルゴリズムの**計算量のオーダーは変わりません**。変わるのは定数倍（と並列に計算できる量）です。
> - この比較は**実行時間**の比較で、**消費電力（電力効率）は測っていません**。

## 8. 学生向け TODO

- [ ] **TODO 1**：CPU・GPU・TPU の各ランタイムでこの Notebook を実行し、CSV を保存する（使えなかったランタイムは「使えなかった」と記録）。
- [ ] **TODO 2**：`VECTOR_SIZES`・`MATRIX_SIZES` を変更する（安全上限に注意）。
- [ ] **TODO 3**：同じ JAX 関数の `compute_s` を CPU と GPU/TPU で比べ、アクセラレータが有利になるサイズを求める。
- [ ] **TODO 4**：`first_call_s` と `compute_s` の比から、「コンパイルの元が取れる」実行回数を計算する。
- [ ] **TODO 5**：転送・取り出しを含めた合計時間で比べ直す。
- [ ] **TODO 6**：`DTYPE` や TPU の行列積精度（`jax_default_matmul_precision`）を変えて、時間と誤差を調べる（Advanced）。

## 9. 考察課題

1. 要素ごとの演算・総和・行列積のうち、アクセラレータの効果が最も大きかったのはどれですか。「計算量とデータ量の比」から説明してください。
2. NumPy（CPU）と JAX（CPU）の差は、何によるものだと考えられますか。
3. JAX の初回の実行が遅いのはなぜですか。実際のアプリケーションでは、この遅さはどのように扱われていると思いますか。
4. 「TPU は速い」と一言で言えないのはなぜですか。この Notebook の結果を使って説明してください。

## 10. 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第6章 ソート）
- 参考 Notebook：`ai-programming-2026-team-project-1-starter/A_cpu_gpu_tpu/A_cpu_gpu_tpu_benchmark/starter.ipynb`（MIT License, Copyright (c) 2026 Daigo Takeuchi）
- NumPy ドキュメント <https://numpy.org/doc/>
- PyTorch ドキュメント：CUDA semantics（非同期実行と同期）<https://pytorch.org/docs/stable/notes/cuda.html>
- JAX ドキュメント：Asynchronous dispatch、Just-in-time compilation <https://docs.jax.dev/>

- E2 リポジトリの `README.md`